### Import and setup

In [16]:
import numpy as np
import pickle
from pathlib import Path
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from collections import Counter
import json
import nltk
from datetime import datetime

# === CONFIG ===
SAVE_DIR = Path("results")
SAVE_DIR.mkdir(exist_ok=True, parents=True)
RANDOM_SEED = 42
N_CLUSTERS = 20  # Align with other pipelines
SAMPLE_SIZE = 20000

np.random.seed(RANDOM_SEED)

### Load shared sample data

In [17]:
with open("shared_data/shared_sample_data.pkl", "rb") as f:
    shared_data = pickle.load(f)

cleaned_texts = shared_data["cleaned_texts"]  # For coherence calculation
original_texts = shared_data.get("original_texts", cleaned_texts)  # Raw for E5 input

print(f"Loaded {len(original_texts)} segments")

Loaded 20000 segments


### Load stopwords

In [18]:
nltk.download('stopwords', quiet=True)

malay_stopwords_path = "stopwords-ms-MannualOp.txt"  
with open(malay_stopwords_path, "r", encoding="utf-8") as f:
    malay_stopwords = [line.strip() for line in f if line.strip()]

english_stopwords = list(nltk.corpus.stopwords.words('english'))
combined_stopwords = english_stopwords + malay_stopwords

def light_clean(text):
    tokens = text.lower().split()
    tokens = [t for t in tokens if t not in combined_stopwords and len(t) > 2]
    return ' '.join(tokens)

processed_texts = [light_clean(t) for t in original_texts]

### Generate E5 embeddings

In [19]:
model = SentenceTransformer('intfloat/multilingual-e5-large', device='cuda')  
embeddings = model.encode(processed_texts, batch_size=32, show_progress_bar=True, normalize_embeddings=True)
np.save(SAVE_DIR / "e5_embeddings.npy", embeddings)
print("E5 embeddings saved.")

Batches: 100%|██████████| 625/625 [01:04<00:00,  9.75it/s]


E5 embeddings saved.


### Compute similarity & distance matrix

In [20]:
sim_matrix = cosine_similarity(embeddings)
dist_matrix = np.clip(1 - sim_matrix, 0, None)
np.fill_diagonal(dist_matrix, 0)  # Important for Agglomerative

### Clustering

In [22]:
clustering = AgglomerativeClustering(
    n_clusters=N_CLUSTERS,
    metric='precomputed',
    linkage='average'
)
labels = clustering.fit_predict(dist_matrix)
np.save(SAVE_DIR / "cluster_labels.npy", labels)
print(f"Clustering completed. Labels shape: {labels.shape}")

Clustering completed. Labels shape: (20000,)


### Evaluation Metrics

In [25]:
# Silhouette
silhouette = silhouette_score(embeddings, labels, metric='cosine')

# Coherence (use cleaned_texts for fair comparison)
dictionary = Dictionary([text.split() for text in cleaned_texts])
texts_tokenized = [text.split() for text in cleaned_texts]

topics = []
for i in range(N_CLUSTERS):
    cluster_indices = np.where(labels == i)[0]
    cluster_docs = [cleaned_texts[j].split() for j in cluster_indices]
    if len(cluster_docs) > 0:
        all_words = [word for doc in cluster_docs for word in doc]
        top_words = [word for word, _ in Counter(all_words).most_common(50)]
        topics.append(top_words)

if not topics:
    topics = [['empty']] * N_CLUSTERS

# C_V (sliding window cosine, stable)
cm_cv = CoherenceModel(
    topics=topics,
    texts=texts_tokenized,
    dictionary=dictionary,
    coherence='c_v'
)

# NPMI
cm_npmi = CoherenceModel(
    topics=topics,
    texts=texts_tokenized,
    dictionary=dictionary,
    coherence='c_npmi'  
)

coherence_cv = cm_cv.get_coherence()
coherence_npmi = cm_npmi.get_coherence()  

# Topic Diversity (不变)
unique_words = len(set(word for topic in topics for word in topic[:10]))
td = unique_words / (N_CLUSTERS * 10) if N_CLUSTERS > 0 else 0

print(f"Silhouette: {silhouette:.4f}")
print(f"C_V: {coherence_cv:.4f}")
print(f"NPMI (c_npmi): {coherence_npmi:.4f}")
print(f"Topic Diversity: {td:.4f}")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Silhouette: 0.0222
C_V: 0.5375
NPMI (c_npmi): 0.0730
Topic Diversity: 0.6450


### Save results

In [26]:
# === 7. Save results ===
Model_DIR = Path("model")
Model_DIR.mkdir(exist_ok=True, parents=True)

results = {
    "pipeline": "06_e5_large_sota",
    "n_clusters": N_CLUSTERS,
    "sample_size": SAMPLE_SIZE,
    "silhouette": float(silhouette),
    "coherence_cv": float(coherence_cv),
    "coherence_npmi": float(coherence_npmi) if not np.isnan(coherence_npmi) else None,
    "topic_diversity": float(td),
    "saved_at": datetime.now().isoformat(),
    "description": "External SOTA: Pure Multilingual-E5-Large embeddings + Agglomerative Clustering on lightly cleaned text (no CPATF, no hybrid)"
}

model_bundle = {
    "embeddings": embeddings,               # numpy array of E5 embeddings
    "cluster_labels": labels,               # numpy array (20000,)
    "results": results,                     # the dict above
    "description": "External SOTA: Multilingual-E5-Large embeddings + Agglomerative Clustering (no trainable params)",
    "saved_at": datetime.now().isoformat(),
    "sample_size": SAMPLE_SIZE,
    "n_clusters": N_CLUSTERS
}

with open(SAVE_DIR / "pipeline6_results.json", "w") as f:
    json.dump(results, f, indent=2)

with open(Model_DIR / "e5_large_sota_model_bundle.pkl", "wb") as f:
    pickle.dump(model_bundle, f)

print("Pipeline 6 completed!")
print("Pipeline 6 model bundle saved to:", Model_DIR / "e5_large_sota_model_bundle.pkl")
print("JSON results saved to:", SAVE_DIR / "pipeline6_results.json")
print(f"Silhouette: {silhouette:.4f}")
print(f"C_V: {coherence_cv:.4f} | NPMI: {coherence_npmi:.4f}")
print(f"Topic Diversity: {td:.4f}")

Pipeline 6 completed!
Pipeline 6 model bundle saved to: model/e5_large_sota_model_bundle.pkl
JSON results saved to: results/pipeline6_results.json
Silhouette: 0.0222
C_V: 0.5375 | NPMI: 0.0730
Topic Diversity: 0.6450
